# Databricks Notebook Logger Demo
## Supports Python & SQL

This notebook demonstrates how to create SAS-style audit trails for Databricks notebooks. The logging system captures all executed code, outputs, SQL queries, warnings, and errors, then securely transfers the log file and HTML snapshot via SFTP to your designated storage location.

### Quick Setup

To use this logging system in your own notebook:

1. Install and import the `notebook-audit-logger` package
2. Call `start_logging(output_directory="path/to/output/directory")` 
3. Place all your analysis code in the following cells
4. Call `stop_logging()` at the end of your notebook
5. Run the entire notebook - your audit trail artifacts will be automatically saved to your Workspace and transferred via SFTP

See the cells below for a complete working example.

### Import Logging Module

Import the logging functions you need:

In [0]:
# Install and import the logging module
# %pip install nb-audit-logger
# from nb_audit_logger import *

import sys
if "/Workspace/Repos/hmcooper@berkeley.edu/Databricks-Notebook-Logger/src" not in sys.path:
    sys.path.insert(0, "/Workspace/Repos/hmcooper@berkeley.edu/Databricks-Notebook-Logger/src")
from nb_audit_logger import *

Credential persistence path chosen: /home/spark-bc702c34-ff81-4240-9ef5-8f/.sftp_saved_creds_hmcooper.json


### Call `start_logging()` Function

This initializes the logging system and prints a summary of key run metadata. You'll be prompted to enter your SFTP credentials to configure secure transfer of the audit trail artifacts.

In [0]:
start_logging(output_directory="/output/logs")

user                 : hmcooper@berkeley.edu
run start timestamp  : 2026-03-07T17:11:36+00:00
notebook             : /Repos/hmcooper@berkeley.edu/Databricks-Notebook-Logger/demo/demo_notebook
log file             : /Repos/hmcooper@berkeley.edu/Databricks-Notebook-Logger/demo/demo_notebook.log
cluster id           : 0307-171125-5mde8fi8-v2n
python               : 3.12.3
ipython              : 8.32.0
spark                : 3.5.2
NOTE: Logging started



SFTP host:  eu-central-1.sftpcloud.io

SFTP port:  22

Username:  e63e355e68b14512990a96f20332cc40

Password (hidden):  [REDACTED]

Validating SFTP credentials...
Connecting to eu-central-1.sftpcloud.io:22 as e63e355e68b14512990a96f20332cc40 ...
Connection established.
Closing connection.
SFTP authentication successful.


Save your credentials for this cluster only (cleared when it restarts)? [y/N]:  y

Credentials saved to /home/spark-bc702c34-ff81-4240-9ef5-8f/.sftp_saved_creds_hmcooper.json (mode 0o600).
SFTP: local log path will be /home/spark-bc702c34-ff81-4240-9ef5-8f/hmcooper/demo_notebook.log


### Analysis Code

All code in this section is automatically recorded in the log file. This includes `df.show()` outputs, `print()` statements, warnings, and errors. Examples below demonstrate logging with sample healthcare data.

In [0]:
# Imports
import pyspark
from pyspark import sql, SparkConf
from pyspark.sql import functions as f
from pyspark.sql.types import *
import os, sys
from pyspark.sql.window import Window
import warnings
from datetime import date

In [0]:
# Create sample patient visit data
visit_data = [
    ("P001", date(2023, 8, 21), "99213", "Hypertension"),
    ("P002", date(2023, 8, 22), "99214", "Diabetes"),
    ("P003", date(2023, 8, 23), "99212", "Annual checkup"),
    ("P001", date(2023, 8, 24), "99215", "Hypertension"),
    ("P004", date(2023, 8, 25), "99213", "Asthma"),
    ("P002", date(2023, 8, 26), "99213", "Diabetes"),
]

patient_visits = spark.createDataFrame(visit_data, ["patient_id", "visit_date", "procedure_code", "diagnosis"])
patient_visits.show()

+----------+----------+--------------+--------------+
|patient_id|visit_date|procedure_code|     diagnosis|
+----------+----------+--------------+--------------+
|      P001|2023-08-21|         99213|  Hypertension|
|      P002|2023-08-22|         99214|      Diabetes|
|      P003|2023-08-23|         99212|Annual checkup|
|      P001|2023-08-24|         99215|  Hypertension|
|      P004|2023-08-25|         99213|        Asthma|
|      P002|2023-08-26|         99213|      Diabetes|
+----------+----------+--------------+--------------+



In [0]:
# Create sample patient demographics table
demographics_data = [
    ("P001", "Male", 45, "Commercial"),
    ("P002", "Female", 62, "Medicare"),
    ("P003", "Male", 34, "Commercial"),
    ("P004", "Female", 28, "Medicaid")
]

demographics = spark.createDataFrame(demographics_data, ["patient_id", "gender", "age", "insurance"])
demographics.show()

+----------+------+---+----------+
|patient_id|gender|age| insurance|
+----------+------+---+----------+
|      P001|  Male| 45|Commercial|
|      P002|Female| 62|  Medicare|
|      P003|  Male| 34|Commercial|
|      P004|Female| 28|  Medicaid|
+----------+------+---+----------+



In [0]:
# Join sample visit with demographic data
patient_analysis = patient_visits.join(demographics, on="patient_id", how="left")
patient_analysis.show()

+----------+----------+--------------+--------------+------+---+----------+
|patient_id|visit_date|procedure_code|     diagnosis|gender|age| insurance|
+----------+----------+--------------+--------------+------+---+----------+
|      P001|2023-08-21|         99213|  Hypertension|  Male| 45|Commercial|
|      P002|2023-08-22|         99214|      Diabetes|Female| 62|  Medicare|
|      P003|2023-08-23|         99212|Annual checkup|  Male| 34|Commercial|
|      P001|2023-08-24|         99215|  Hypertension|  Male| 45|Commercial|
|      P004|2023-08-25|         99213|        Asthma|Female| 28|  Medicaid|
|      P002|2023-08-26|         99213|      Diabetes|Female| 62|  Medicare|
+----------+----------+--------------+--------------+------+---+----------+



In [0]:
# Create sample summary statistics by demographics
summary = patient_analysis.groupBy("gender", "insurance") \
    .agg(
        f.count("visit_date").alias("total_visits"),
        f.countDistinct("patient_id").alias("unique_patients"),
        f.avg("age").alias("avg_age")
    ) \
    .orderBy("gender", "insurance")

summary.show(truncate=False)

+------+----------+------------+---------------+------------------+
|gender|insurance |total_visits|unique_patients|avg_age           |
+------+----------+------------+---------------+------------------+
|Female|Medicaid  |1           |1              |28.0              |
|Female|Medicare  |2           |1              |62.0              |
|Male  |Commercial|3           |2              |41.333333333333336|
+------+----------+------------+---------------+------------------+



In [0]:
# Test print statement
print("Analysis complete: Generated summary statistics by gender and insurance type.")

Analysis complete: Generated summary statistics by gender and insurance type.


### Logging SQL Queries and Output

* To log SQL code, run it using `spark.sql("SQL QUERY HERE")`. The query will appear in the log file
* Spark is lazy—queries won't execute unless an action is called. Use `.show()` to execute and display results
* Use `.show(truncate=False)` to prevent values from being cut off in the log file

In [0]:
# Create temporary view to query with SQL
patient_visits.createOrReplaceTempView("patient_visits")

In [0]:
# Query from spark.sql() will show up in the log
spark.sql("""
create or replace temp view diagnosis_summary as
select diagnosis, count(*) as patient_count
from patient_visits
group by diagnosis
""")

DataFrame[]

In [0]:
# Spark is lazy: doesn't actually execute this query - output will NOT display in the log
spark.sql("""
select count(*) as total_diagnoses
from diagnosis_summary
""")

DataFrame[total_diagnoses: bigint]

In [0]:
# Use .show(truncate=False) to execute query and display output in the log file
# truncate=False ensures no values are truncated/cut off in the log file
spark.sql("""
select *
from diagnosis_summary
where diagnosis = 'Hypertension'
    or diagnosis = 'Diabetes'
""").show(truncate=False)

+------------+-------------+
|diagnosis   |patient_count|
+------------+-------------+
|Hypertension|2            |
|Diabetes    |2            |
+------------+-------------+



### Logging Spark Table Contents

`log_df(table_name, limit)` displays Spark table contents in both the notebook and the log file. Pass the table name directly (e.g. `"patient_visits"`) or use `schema.table` format for permanent tables. The row limit defaults to 20.

In [0]:
# Use log_df to display table contents in the log file
log_df("diagnosis_summary", limit=10)

DataFrame[diagnosis: string, patient_count: bigint]

[LOG] Showing table: diagnosis_summary
+--------------+-------------+
|diagnosis     |patient_count|
+--------------+-------------+
|Hypertension  |2            |
|Diabetes      |2            |
|Annual checkup|1            |
|Asthma        |1            |
+--------------+-------------+



### Warnings and Errors in the Log

All warnings and errors are automatically captured and written to the log file. See `error_notebook` for a full example of error handling in action.

In [0]:
# User warning
warnings.warn("This is a test warning", UserWarning)

### Call `stop_logging()` Function

This finalizes the log and securely transfers both the log file and HTML snapshot to your designated storage location via SFTP.

In [0]:
stop_logging()


NOTEBOOK RAN SUCCESSFULLY WITH NO ERRORS
run end timestamp    : 2026-03-07T17:12:43+00:00
total runtime        : 67.61 seconds
NOTE: Logging stopped. Uploaded to Workspace → /Repos/hmcooper@berkeley.edu/Databricks-Notebook-Logger/demo/demo_notebook.log
SFTP: wrote local log → /home/spark-bc702c34-ff81-4240-9ef5-8f/hmcooper/demo_notebook.log
Opening SFTP session...
Connecting to eu-central-1.sftpcloud.io:22 as e63e355e68b14512990a96f20332cc40 ...
Connection established.
Ensuring remote directory exists: /output/logs
Created directory: /output
Created directory: /output/logs
Uploading file:
 remote: /output/logs/demo_notebook.log
Upload completed successfully.
Closing connection.
SFTP: uploaded → /output/logs/demo_notebook.log
SFTP: deleted local log → /home/spark-bc702c34-ff81-4240-9ef5-8f/hmcooper/demo_notebook.log
SFTP: wrote local HTML → /home/spark-bc702c34-ff81-4240-9ef5-8f/hmcooper/demo_notebook.html
Opening SFTP session...
Connecting to eu-central-1.sftpcloud.io:22 as e63e355e68

## Logging System
### Key Features

- Header with notebook metadata displayed in both the notebook and log file
- Trailer indicating whether the notebook ran successfully, along with any warnings captured during the run
- Upon any cell error, the notebook stops running immediately and the log file is uploaded with a full error summary
- All executed code is captured in the log alongside any `print()` and `df.show()` output
- Full support for SQL code and query output logging
- `log_df()` helper function for displaying Spark table contents directly in the log file
- Per-cell and total notebook runtimes recorded in the log
- Clean, readable log format with `NOTE:` runtime annotations and cell separators
- Each run automatically overwrites the previous log file
- Log file and HTML snapshot of the notebook are automatically transferred to your designated SFTP location at the end of every run

### How It Works

This module hooks into IPython and Databricks to automatically track everything that happens during a notebook run, saving it as a clean, readable log file. For a visual overview of how the logging system works on the backend, refer to *[databricks_log_diagram.pdf](databricks_log_diagram.pdf)*. The PDF may not render clearly in GitLab, so you will need to download it to zoom in

1. **In-memory buffer** — A `StringIO` buffer acts as an in-memory text file, accumulating everything that happens during the run and flushing to the log file when the notebook finishes or encounters an error

2. **IPython event hooks** — `pre_run_cell` and `post_run_cell` hooks trigger before and after every cell, capturing cell source code, outputs, warnings, and timings in the exact order of execution

3. **Output capture** — `sys.stdout` and `sys.stderr` are redirected to a `Tee` object that writes simultaneously to the notebook UI and the log buffer

4. **Error handling** — If any cell fails, the logger detects the error, appends a full error summary to the log, and stops the run

5. **Workspace upload** — On run completion or error, the log buffer is uploaded directly to the Databricks Workspace via the built-in REST API. This approach was chosen because Databricks does not behave like a standard file system, and many enterprise environments restrict direct file access on the underlying machine

6. **SFTP transfer** — The log file and an HTML snapshot of the notebook are securely transferred to your designated remote directory via SFTP. Credentials are prompted once when the logger is initalized and cached for the remainder of the session

## Limitations
* **Markdown cells**: Not captured in the log (not executed by the Python kernel)
* **Cell titles**: Not included in the log (part of the notebook interface, not executable code)